## Setup

In [1]:
#Import lib
import os
import scanpy as sc
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

import colorcet as cc
colors10 = cc.palette["glasbey"][10:20]
colors80 = cc.palette["glasbey"][:80]

/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_mtx from `anndata` is deprecated. Import anndata.io.read_mtx instead.
  warnings.warn(msg, FutureWarning)
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Impor

In [2]:
# Set scanpy plotting defaults
sc.settings.set_figure_params(
    dpi=300,
    dpi_save=300,
    figsize=(3, 2),
    facecolor='white',
    fontsize=7
)

## TERRA finetuned

###  Merge embeddings

In [3]:
# Define the base directory
base_dir = Path("/lustre/scratch126/cellgen/lotfollahi/zh4/NicheJEPA/Data/Visium_Embryo_Figure/Zero_Shot_Inference/Final/Finetuned")

# Set scanpy figure directory
sc.settings.figdir = base_dir 

In [4]:
# Concatenate h5ad files
h5ad_files = list(base_dir.glob("*.h5ad"))

adatas = []
sample_names = []

for h5ad_file in sorted(h5ad_files):
    if 'ZS_merged' in h5ad_file.name:
        continue

    filename = h5ad_file.stem
    parts = filename.split('_')
    sample_name = '_'.join([p for p in parts if p not in ['Visium', 'Embryo', 'adata', 'embeddings']])

    adata = sc.read_h5ad(h5ad_file)

    adata.obs['sample'] = sample_name
    adatas.append(adata)
    sample_names.append(sample_name)

print(f"\nLoaded {len(adatas)} samples")
print(f"Sample names: {sample_names}")


Loaded 10 samples
Sample names: ['section_040', 'section_052', 'section_070', 'section_086', 'section_111', 'section_134', 'section_151', 'section_177', 'section_205', 'section_251']


In [5]:
# Merge embeddings into single anndata
import anndata as ad

# Make variable names unique for each anndata object
for adata in adatas:
    adata.var_names_make_unique()

# Concatenate all anndata objects
adata_merged = ad.concat(adatas, join='inner', merge='unique')

print(f"Merged shape: {adata_merged.shape}")


Merged shape: (60212, 9180)


In [6]:
# Filter out cells where 'C_scANVI' is 'NA' or NaN
adata_merged = adata_merged[~adata_merged.obs['C_scANVI'].isna() & (adata_merged.obs['C_scANVI'] != 'NA')]

print(f"After filtering: {adata_merged.shape[0]} cells remaining")

After filtering: 51784 cells remaining


In [7]:
# Save the merged file
output_path = base_dir / "ZS_merged_embeddings.h5ad"
adata_merged.write_h5ad(output_path)
print(f"\nSaved merged embeddings to: {output_path}")

/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/anndata/_core/anndata.py:1138: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c



Saved merged embeddings to: /lustre/scratch126/cellgen/lotfollahi/zh4/NicheJEPA/Data/Visium_Embryo_Figure/Zero_Shot_Inference/Final/Finetuned/ZS_merged_embeddings.h5ad


### UMAP cell embedding

In [8]:
# Compute neighbors and UMAP using spot_emb
sc.pp.neighbors(adata_merged, use_rep='cell_emb', n_neighbors=15)
sc.tl.umap(
    adata_merged,
    min_dist=0.5,      
    spread=1.0
)

### Batch integration

In [9]:
# Plot UMAP by sample
sc.pl.umap(
    adata_merged,
    color='sample',
    palette=colors10,
    show=False,
    legend_loc=None,
    size=1,
    alpha=0.8,
    frameon=False,
    save='finetuned_spot_emb_sample_no_legend.svg'
)

plt.tight_layout()
plt.show()

/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


<Figure size 900x600 with 0 Axes>

In [10]:
# Plot UMAP by sample
sc.pl.umap(
    adata_merged,
    color='sample',
    palette=colors10,
    show=False,
    legend_loc='right margin',
    size=1,
    alpha=0.8,
    frameon=False,
    save='finetuned_spot_emb_sample_with_legend.svg'
)

plt.tight_layout()
plt.show()

/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


<Figure size 900x600 with 0 Axes>

### Ground-truth annotation

In [11]:
# Plot UMAP by annnotation
sc.pl.umap(
    adata_merged,
    color='C_scANVI',
    palette=colors80,
    show=False,
    legend_loc=None,
    size=1,
    alpha=0.8,
    frameon=False,
    save='finetuned_spot_emb_C_scANVI_no_legend.svg'
)

plt.tight_layout()
plt.show()

/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


<Figure size 900x600 with 0 Axes>

In [12]:
# Plot UMAP by annnotation
sc.pl.umap(
    adata_merged,
    color='C_scANVI',
    palette=colors80,
    show=False,
    legend_loc='right margin',
    size=1,
    alpha=0.8,
    frameon=False,
    save='finetuned_spot_emb_C_scANVI_with_legend.svg'
)

plt.tight_layout()
plt.show()

/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


<Figure size 900x600 with 0 Axes>

### Group ground truth annotation

In [13]:
# Plot UMAP colored by ground truth annotation 
# Create a new column for highlighting
adata_merged.obs['highlight_spots'] = adata_merged.obs['C_scANVI'].astype(str)

#Brain regions
Neural_tube = [
    'Diencephalon_level2',
    'Hindbrain_level2',
    'Isthmus_level2',
    'Medulla oblongata_level2',
    'Midbrain_level1',
    'Pons_level2',
    'Rhombomere_level2',
    'Telencephalon_level2',
    'Spinal Cord_level0',
]

#Heart regions
heart = [
    'Left Atrium_level1',
    'Left Ventricle_level1',
    'Right Atrium_level1',
    'Right Ventricle_level1',
]

#Mark cells of interest
adata_merged.obs.loc[adata_merged.obs['highlight_spots'].isin(Neural_tube), 'highlight_spots'] = 'Neural tube'
adata_merged.obs.loc[adata_merged.obs['highlight_spots'].isin(heart), 'highlight_spots'] = 'Heart'
adata_merged.obs.loc[adata_merged.obs['highlight_spots'] == 'Liver_level0', 'highlight_spots'] = 'Liver'
adata_merged.obs.loc[~adata_merged.obs['highlight_spots'].isin(['Neural tube', 'Liver', 'Heart']), 'highlight_spots'] = 'Other'
# Create palette: grey for 'Other', distinct color for cells of interest
highlight_palette = {
    'Other': '#D3D3D3',      # Light grey
    'Liver': '#9966FF',  # Purple
    'Heart': '#CC0066',  # Red
    'Neural tube': '#66B3FF'       # blue
}

sc.pl.umap(
    adata_merged, 
    color='highlight_spots',
    palette=highlight_palette,
    show=False, 
    legend_loc=None,
    size=1,
    alpha=0.8,
    frameon=False,
    save='finetuned_spot_emb_highlight_spots_no_legend.svg'
)
plt.tight_layout()
plt.show()

/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


<Figure size 900x600 with 0 Axes>

In [14]:
sc.pl.umap(
    adata_merged, 
    color='highlight_spots',
    palette=highlight_palette,
    show=False, 
    legend_loc='right margin',
    size=1,
    alpha=0.8,
    frameon=False,
    save='finetuned_spot_emb_highlight_spots_with_legend.svg'
)
plt.tight_layout()
plt.show()

/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


<Figure size 900x600 with 0 Axes>

### Leiden clustering

In [15]:
#Leiden cluster
sc.tl.leiden(adata_merged, 
                 key_added=f'emb_leiden',
                 resolution=0.15)

# Plot UMAP with best resolution
sc.pl.umap(
    adata_merged, 
    color='emb_leiden',
    palette=colors80, 
    show=False, 
    legend_loc='right margin',
    size=1,
    alpha=0.8,
    frameon=False,
    save='finetuned_spot_emb_emb_leiden_with_legend.svg'
)
plt.tight_layout()
plt.show()

/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


<Figure size 900x600 with 0 Axes>

In [16]:
# Plot UMAP with best resolution
sc.pl.umap(
    adata_merged, 
    color='emb_leiden',
    palette=colors80, 
    show=False, 
    legend_loc=None,
    size=1,
    alpha=0.8,
    frameon=False,
    save='finetuned_spot_emb_emb_leiden_no_legend.svg'
)
plt.tight_layout()
plt.show()

/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


<Figure size 900x600 with 0 Axes>

### Group Leiden Clusters

In [17]:
#Subset Leiden Clusters
adata_merged.obs['highlight_leiden'] = adata_merged.obs['emb_leiden'].astype(str)

#Define clusters to highlight
adata_merged.obs['highlight_leiden'] = 'Other'
adata_merged.obs.loc[adata_merged.obs['emb_leiden'].astype(str) == '7', 'highlight_leiden'] = 'Cluster 7'
adata_merged.obs.loc[adata_merged.obs['emb_leiden'].astype(str) == '8', 'highlight_leiden'] = 'Cluster 8'
adata_merged.obs.loc[adata_merged.obs['emb_leiden'].astype(str).isin(['4','6','9']), 'highlight_leiden'] = 'Clusters 4, 6 and 9'

# Create palette
highlight_leiden_palette = {
    'Other': '#D3D3D3',  
    'Cluster 7': '#CC0066',       
    'Cluster 8': '#9966FF',       
    'Clusters 4, 6 and 9': '#66B3FF'      
}

sc.pl.umap(
    adata_merged, 
    color='highlight_leiden',
    palette=highlight_leiden_palette,
    show=False, 
    legend_loc=None,
    size=1,
    alpha=0.8,
    frameon=False,
    save='finetuned_spot_emb_highlight_leiden_no_legend.svg'
)
plt.tight_layout()
plt.show()

/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


<Figure size 900x600 with 0 Axes>

In [18]:
sc.pl.umap(
    adata_merged, 
    color='highlight_leiden',
    palette=highlight_leiden_palette,
    show=False, 
    legend_loc='right margin',
    size=1,
    alpha=0.8,
    frameon=False,
    save='finetuned_spot_emb_highlight_leiden_with_legend.svg'
)
plt.tight_layout()
plt.show()

/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


<Figure size 900x600 with 0 Axes>

### Spatial maps of Leiden clusters

In [19]:
# Define specific samples to plot
samples_to_plot = adata_merged.obs['sample'].unique().tolist()

# Create output directory for plots
output_dir = base_dir / "spatial_plots"
output_dir.mkdir(exist_ok=True)

# Set scanpy figure directory
sc.settings.figdir = output_dir

for sample_name in samples_to_plot:
    
    # Subset to current sample
    adata_sample = adata_merged[adata_merged.obs['sample'] == sample_name].copy()
    

    sc.pl.spatial(
        adata_sample,
        img_key=None,
        color='highlight_leiden',
        palette=highlight_leiden_palette,
        spot_size=300,
        alpha=1,
        legend_loc=None,
        show=False,
        frameon=False,
        save=f'_{sample_name}_leiden_clusters.svg'
    )

    sc.pl.spatial(
        adata_sample,
        img_key=None,
        color='highlight_leiden',
        palette=highlight_leiden_palette,
        spot_size=300,
        alpha=1,
        legend_loc='right margin',
        show=False,
        frameon=False,
        save=f'_{sample_name}_leiden_clusters_key.svg'
    )

/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))


/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))


/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))


/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))


/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))


/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))


... (7 outputs left out)

/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))


/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))


/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))


/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))


### Calculate ARI and NMI metrics

In [20]:
# Leiden resolution with best ARI 
latent_leiden_resolution = 0.15

# Compute metrics over different resolutions
nmi_scores = []
ari_scores = []
n_cell_types_list = []

latent_leiden_resolution_list = [latent_leiden_resolution,
                                 latent_leiden_resolution-0.04,
                                 latent_leiden_resolution-0.03,
                                 latent_leiden_resolution-0.02,
                                 latent_leiden_resolution-0.01,
                                 latent_leiden_resolution+0.01,
                                 latent_leiden_resolution+0.02,
                                 latent_leiden_resolution+0.03,
                                 latent_leiden_resolution+0.04]

for i, latent_leiden_resolution in enumerate(latent_leiden_resolution_list[1:]):

    sc.tl.leiden(adata_merged,
                 key_added=f'emb_leiden',
                 resolution=latent_leiden_resolution)

    n_cell_types_list.append(adata_merged.obs[f'emb_leiden'].nunique())

    labels_true = adata_merged.obs['C_scANVI']
    labels_pred = adata_merged.obs['emb_leiden']
    
    nmi_score = normalized_mutual_info_score(labels_true, labels_pred, average_method="arithmetic")
    ari_score = adjusted_rand_score(labels_true, labels_pred)

    nmi_scores.append(nmi_score)
    ari_scores.append(ari_score)


print(f"NMI scores: {nmi_scores}.")
print(f"ARI scores: {ari_scores}.")
print(f"Number of clusters: {n_cell_types_list}.")

NMI scores: [0.5050771163744487, 0.5140834323012311, 0.5010747464125674, 0.48371014783581445, 0.5037402102479521, 0.5020439572998047, 0.5095513744304463, 0.49043647272143737].
ARI scores: [0.30940545304585804, 0.35062613619147914, 0.3463592866650655, 0.2968677862399142, 0.33753136849181786, 0.3486178483053798, 0.3376972431748891, 0.273985660604052].
Number of clusters: [9, 11, 11, 11, 14, 14, 15, 15].


### UMAP Neighborhood embedding

In [21]:
# Set scanpy figure directory
sc.settings.figdir = base_dir 

In [22]:
# Compute neighbors and UMAP using neighborhood_emb
sc.pp.neighbors(adata_merged, use_rep='neighborhood_emb', n_neighbors=15)
sc.tl.umap(
    adata_merged,
    min_dist=0.5,      
    spread=1.0
)

### Batch integration

In [23]:
# Plot UMAP by sample
sc.pl.umap(
    adata_merged,
    color='sample',
    palette=colors10,
    show=False,
    legend_loc=None,
    size=1,
    alpha=0.8,
    frameon=False,
    save='finetuned_neighborhood_emb_sample_no_legend.svg'
)

plt.tight_layout()
plt.show()

/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


<Figure size 900x600 with 0 Axes>

In [24]:
# Plot UMAP by sample
sc.pl.umap(
    adata_merged,
    color='sample',
    palette=colors10,
    show=False,
    legend_loc='right margin',
    size=1,
    alpha=0.8,
    frameon=False,
    save='finetuned_neighborhood_emb_sample_with_legend.svg'
)

plt.tight_layout()
plt.show()

/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


<Figure size 900x600 with 0 Axes>

### Ground-truth annotation

In [25]:
# Plot UMAP by annnotation
sc.pl.umap(
    adata_merged,
    color='C_scANVI',
    palette=colors80,
    show=False,
    legend_loc=None,
    size=1,
    alpha=0.8,
    frameon=False,
    save='finetuned_neighborhood_emb_C_scANVI_no_legend.svg'
)

plt.tight_layout()
plt.show()

/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


<Figure size 900x600 with 0 Axes>

In [26]:
# Plot UMAP by annnotation
sc.pl.umap(
    adata_merged,
    color='C_scANVI',
    palette=colors80,
    show=False,
    legend_loc='right margin',
    size=1,
    alpha=0.8,
    frameon=False,
    save='finetuned_neighborhood_emb_C_scANVI_with_legend.svg'
)

plt.tight_layout()
plt.show()

/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


<Figure size 900x600 with 0 Axes>

### Group ground truth annotation

In [27]:
# Create a new column for highlighting
adata_merged.obs['highlight_spots'] = adata_merged.obs['C_scANVI'].astype(str)

#Brain regions
Neural_tube = [
    'Diencephalon_level2',
    'Hindbrain_level2',
    'Isthmus_level2',
    'Medulla oblongata_level2',
    'Midbrain_level1',
    'Pons_level2',
    'Rhombomere_level2',
    'Telencephalon_level2',
    'Spinal Cord_level0',
]

#Heart regions
heart = [
    'Left Atrium_level1',
    'Left Ventricle_level1',
    'Right Atrium_level1',
    'Right Ventricle_level1',
]

#Mark cells of interest
adata_merged.obs.loc[adata_merged.obs['highlight_spots'].isin(Neural_tube), 'highlight_spots'] = 'Neural tube'
adata_merged.obs.loc[adata_merged.obs['highlight_spots'].isin(heart), 'highlight_spots'] = 'Heart'
adata_merged.obs.loc[adata_merged.obs['highlight_spots'] == 'Liver_level0', 'highlight_spots'] = 'Liver'
adata_merged.obs.loc[~adata_merged.obs['highlight_spots'].isin(['Neural tube', 'Liver', 'Heart']), 'highlight_spots'] = 'Other'

# Create palette
highlight_palette = {
    'Other': '#D3D3D3',     
    'Liver': '#9966FF',  
    'Heart': '#CC0066', 
    'Neural tube': '#66B3FF'       
}

sc.pl.umap(
    adata_merged, 
    color='highlight_spots',
    palette=highlight_palette,
    show=False, 
    legend_loc=None,
    size=1,
    alpha=0.8,
    frameon=False,
    save='finetuned_neighborhood_emb_highlight_spots_no_legend.svg'
)

plt.tight_layout()
plt.show()

/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


<Figure size 900x600 with 0 Axes>

In [28]:
sc.pl.umap(
    adata_merged, 
    color='highlight_spots',
    palette=highlight_palette,
    show=False, 
    legend_loc='right margin',
    size=1,
    alpha=0.8,
    frameon=False,
    save='finetuned_neighborhood_emb_highlight_spots_with_legend.svg'
)

plt.tight_layout()
plt.show()

/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


<Figure size 900x600 with 0 Axes>

## TERRA non-finetuned

###  Merge embeddings

In [29]:
# Define the base directory
base_dir = Path("/lustre/scratch126/cellgen/lotfollahi/zh4/NicheJEPA/Data/Visium_Embryo_Figure/Zero_Shot_Inference/Final/Non_finetuned")

# Set scanpy figure directory
sc.settings.figdir = base_dir 

In [30]:
# Concatenate h5ad files
h5ad_files = list(base_dir.glob("*.h5ad"))

adatas = []
sample_names = []

for h5ad_file in sorted(h5ad_files):
    if 'ZS_merged' in h5ad_file.name:
        continue

    filename = h5ad_file.stem
    parts = filename.split('_')
    sample_name = '_'.join([p for p in parts if p not in ['Visium', 'Embryo', 'adata', 'embeddings']])

    adata = sc.read_h5ad(h5ad_file)

    adata.obs['sample'] = sample_name
    adatas.append(adata)
    sample_names.append(sample_name)

print(f"\nLoaded {len(adatas)} samples")
print(f"Sample names: {sample_names}")


Loaded 10 samples
Sample names: ['section_040', 'section_052', 'section_070', 'section_086', 'section_111', 'section_134', 'section_151', 'section_177', 'section_205', 'section_251']


In [31]:
# Merge embeddings into single anndata
import anndata as ad

# Make variable names unique for each anndata object
for adata in adatas:
    adata.var_names_make_unique()

# Concatenate all anndata objects
adata_merged = ad.concat(adatas, join='inner', merge='unique')

print(f"Merged shape: {adata_merged.shape}")


Merged shape: (60212, 9180)


In [32]:
# Filter out cells where 'C_scANVI' is 'NA' or NaN
adata_merged = adata_merged[~adata_merged.obs['C_scANVI'].isna() & (adata_merged.obs['C_scANVI'] != 'NA')]

print(f"After filtering: {adata_merged.shape[0]} cells remaining")

After filtering: 51784 cells remaining


In [33]:
# Save the merged file
output_path = base_dir / "ZS_merged_embeddings.h5ad"
adata_merged.write_h5ad(output_path)
print(f"\nSaved merged embeddings to: {output_path}")

/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/anndata/_core/anndata.py:1138: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c



Saved merged embeddings to: /lustre/scratch126/cellgen/lotfollahi/zh4/NicheJEPA/Data/Visium_Embryo_Figure/Zero_Shot_Inference/Final/Non_finetuned/ZS_merged_embeddings.h5ad


### UMAP cell embedding

In [34]:
# Compute neighbors and UMAP using spot_emb
sc.pp.neighbors(adata_merged, use_rep='cell_emb', n_neighbors=15)
sc.tl.umap(
    adata_merged,
    min_dist=0.5,      
    spread=1.0
)

### Batch integration

In [35]:
# Plot UMAP by sample
sc.pl.umap(
    adata_merged,
    color='sample',
    palette=colors10,
    show=False,
    legend_loc=None,
    size=1,
    alpha=0.8,
    frameon=False,
    save='nonfinetuned_spot_emb_sample_no_legend.svg'
)

plt.tight_layout()
plt.show()

/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


<Figure size 900x600 with 0 Axes>

In [36]:
# Plot UMAP by sample
sc.pl.umap(
    adata_merged,
    color='sample',
    palette=colors10,
    show=False,
    legend_loc='right margin',
    size=1,
    alpha=0.8,
    frameon=False,
    save='nonfinetuned_spot_emb_sample_with_legend.svg'
)

plt.tight_layout()
plt.show()

/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


<Figure size 900x600 with 0 Axes>

### Ground-truth annotation

In [37]:
# Plot UMAP by annnotation
sc.pl.umap(
    adata_merged,
    color='C_scANVI',
    palette=colors80,
    show=False,
    legend_loc=None,
    size=1,
    alpha=0.8,
    frameon=False,
    save='nonfinetuned_spot_emb_C_scANVI_no_legend.svg'
)

plt.tight_layout()
plt.show()

/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


<Figure size 900x600 with 0 Axes>

In [38]:
# Plot UMAP by annnotation
sc.pl.umap(
    adata_merged,
    color='C_scANVI',
    palette=colors80,
    show=False,
    legend_loc='right margin',
    size=1,
    alpha=0.8,
    frameon=False,
    save='nonfinetuned_spot_emb_C_scANVI_with_legend.svg'
)

plt.tight_layout()
plt.show()

/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


<Figure size 900x600 with 0 Axes>

### Group ground truth annotation

In [39]:
# Create a new column for highlighting
adata_merged.obs['highlight_spots'] = adata_merged.obs['C_scANVI'].astype(str)

#Brain regions
Neural_tube = [
    'Diencephalon_level2',
    'Hindbrain_level2',
    'Isthmus_level2',
    'Medulla oblongata_level2',
    'Midbrain_level1',
    'Pons_level2',
    'Rhombomere_level2',
    'Telencephalon_level2',
    'Spinal Cord_level0',
]

#Heart regions
heart = [
    'Left Atrium_level1',
    'Left Ventricle_level1',
    'Right Atrium_level1',
    'Right Ventricle_level1',
]

#Mark cells of interest
adata_merged.obs.loc[adata_merged.obs['highlight_spots'].isin(Neural_tube), 'highlight_spots'] = 'Neural tube'
adata_merged.obs.loc[adata_merged.obs['highlight_spots'].isin(heart), 'highlight_spots'] = 'Heart'
adata_merged.obs.loc[adata_merged.obs['highlight_spots'] == 'Liver_level0', 'highlight_spots'] = 'Liver'
adata_merged.obs.loc[~adata_merged.obs['highlight_spots'].isin(['Neural tube', 'Liver', 'Heart']), 'highlight_spots'] = 'Other'

# Create palette
highlight_palette = {
    'Other': '#D3D3D3',      
    'Liver': '#9966FF',  
    'Heart': '#CC0066', 
    'Neural tube': '#66B3FF'       
}

sc.pl.umap(
    adata_merged, 
    color='highlight_spots',
    palette=highlight_palette,
    show=False, 
    legend_loc=None,
    size=1,
    alpha=0.8,
    frameon=False,
    save='nonfinetuned_spot_emb_highlight_spots_no_legend.svg'
)

plt.tight_layout()
plt.show()

/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


<Figure size 900x600 with 0 Axes>

In [40]:
sc.pl.umap(
    adata_merged, 
    color='highlight_spots',
    palette=highlight_palette,
    show=False, 
    legend_loc='right margin',
    size=1,
    alpha=0.8,
    frameon=False,
    save='nonfinetuned_spot_emb_highlight_spots_with_legend.svg'
)

plt.tight_layout()
plt.show()

/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


<Figure size 900x600 with 0 Axes>

### Leiden clustering

In [41]:
# Use the resolution with best ARI as the final clustering

sc.tl.leiden(adata_merged, 
                 key_added=f'emb_leiden',
                 resolution=0.55)

# Plot UMAP with best resolution
sc.pl.umap(
    adata_merged, 
    color='emb_leiden',
    palette=colors80, 
    show=False, 
    legend_loc='right margin',
    size=1,
    alpha=0.8,
    frameon=False,
    save='nonfinetuned_spot_emb_emb_leiden_with_legend.svg'
)

plt.tight_layout()
plt.show()

/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


<Figure size 900x600 with 0 Axes>

In [42]:
# Plot UMAP with best resolution
sc.pl.umap(
    adata_merged, 
    color='emb_leiden',
    palette=colors80, 
    show=False, 
    legend_loc=None,
    size=1,
    alpha=0.8,
    frameon=False,
    save='nonfinetuned_spot_emb_emb_leiden_no_legend.svg'
)

plt.tight_layout()
plt.show()

/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


<Figure size 900x600 with 0 Axes>

### Calculate ARI and NMI metrics

In [43]:
# Leiden resolution with best ARI 
latent_leiden_resolution = 0.55

# Compute metrics over different resolutions
nmi_scores = []
ari_scores = []
n_cell_types_list = []

latent_leiden_resolution_list = [latent_leiden_resolution,
                                 latent_leiden_resolution-0.04,
                                 latent_leiden_resolution-0.03,
                                 latent_leiden_resolution-0.02,
                                 latent_leiden_resolution-0.01,
                                 latent_leiden_resolution+0.01,
                                 latent_leiden_resolution+0.02,
                                 latent_leiden_resolution+0.03,
                                 latent_leiden_resolution+0.04]

for i, latent_leiden_resolution in enumerate(latent_leiden_resolution_list[1:]):

    sc.tl.leiden(adata_merged,
                 key_added=f'emb_leiden',
                 resolution=latent_leiden_resolution)

    n_cell_types_list.append(adata_merged.obs[f'emb_leiden'].nunique())

    labels_true = adata_merged.obs['C_scANVI']
    labels_pred = adata_merged.obs['emb_leiden']
    
    nmi_score = normalized_mutual_info_score(labels_true, labels_pred, average_method="arithmetic")
    ari_score = adjusted_rand_score(labels_true, labels_pred)

    nmi_scores.append(nmi_score)
    ari_scores.append(ari_score)


print(f"NMI scores: {nmi_scores}.")
print(f"ARI scores: {ari_scores}.")
print(f"Number of clusters: {n_cell_types_list}.")

NMI scores: [0.4804578079053843, 0.49064296128111357, 0.49264511731457167, 0.47994645901133454, 0.4796561716094257, 0.49118522506041096, 0.4909704985593584, 0.4822734706478387].
ARI scores: [0.26929481331161753, 0.2845587839476557, 0.28822098734809404, 0.26542739565233686, 0.2570360989451857, 0.28365973385020804, 0.2576828191950327, 0.2829311490631934].
Number of clusters: [24, 25, 23, 22, 27, 26, 27, 28].


### UMAP Neighborhood embedding

In [44]:
# Set scanpy figure directory
sc.settings.figdir = base_dir 

In [45]:
# Compute neighbors and UMAP using neighborhood_emb
sc.pp.neighbors(adata_merged, use_rep='neighborhood_emb', n_neighbors=15)
sc.tl.umap(
    adata_merged,
    min_dist=0.5,      
    spread=1.0
)

### Batch integration

In [46]:
# Plot UMAP by sample
sc.pl.umap(
    adata_merged,
    color='sample',
    palette=colors10,
    show=False,
    legend_loc=None,
    size=1,
    alpha=0.8,
    frameon=False,
    save='nonfinetuned_neighborhood_emb_sample_no_legend.svg'
)

plt.tight_layout()
plt.show()

/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


<Figure size 900x600 with 0 Axes>

In [47]:
# Plot UMAP by sample
sc.pl.umap(
    adata_merged,
    color='sample',
    palette=colors10,
    show=False,
    legend_loc='right margin',
    size=1,
    alpha=0.8,
    frameon=False,
    save='nonfinetuned_neighborhood_emb_sample_with_legend.svg'
)

plt.tight_layout()
plt.show()

/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


<Figure size 900x600 with 0 Axes>

### Ground-truth annotation

In [48]:
# Plot UMAP by annnotation
sc.pl.umap(
    adata_merged,
    color='C_scANVI',
    palette=colors80,
    show=False,
    legend_loc=None,
    size=1,
    alpha=0.8,
    frameon=False,
    save='nonfinetuned_neighborhood_emb_C_scANVI_no_legend.svg'
)

plt.tight_layout()
plt.show()

/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


<Figure size 900x600 with 0 Axes>

In [49]:
# Plot UMAP by annnotation
sc.pl.umap(
    adata_merged,
    color='C_scANVI',
    palette=colors80,
    show=False,
    legend_loc='right margin',
    size=1,
    alpha=0.8,
    frameon=False,
    save='nonfinetuned_neighborhood_emb_C_scANVI_with_legend.svg'
)

plt.tight_layout()
plt.show()

/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


<Figure size 900x600 with 0 Axes>

## scGPT spatial embeddings

In [50]:
# Define the base directory
base_dir = Path("/lustre/scratch126/cellgen/lotfollahi/zh4/NicheJEPA/Data/Visium_Embryo_Figure/Zero_Shot_Inference/Final/scGPT_spatial")

# Set scanpy figure directory
sc.settings.figdir = base_dir 

In [51]:
#Load scGPT embeddings 
adata_scGPT = sc.read_h5ad("/lustre/scratch126/cellgen/lotfollahi/zh4/NicheJEPA/Data/Visium_Embryo_Figure/Zero_Shot_Inference/scGPT/scGPT_spatial_embeds.h5ad")



### Generate UMAP

In [52]:
# Compute neighbors and UMAP using cell_emb
sc.pp.neighbors(adata_scGPT, use_rep='X', n_neighbors=15)
sc.tl.umap(
    adata_scGPT,
    min_dist=0.5,      
    spread=1.0
)

### Batch integration

In [53]:
# Plot UMAP by sample
sc.pl.umap(
    adata_scGPT,
    color='sample',
    palette=colors10,
    show=False,
    legend_loc=None,
    size=1,
    alpha=0.8,
    frameon=False,
    save='scgpt_sample_no_legend.svg'
)

plt.tight_layout()
plt.show()

/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


<Figure size 900x600 with 0 Axes>

In [54]:
# Plot UMAP by sample
sc.pl.umap(
    adata_scGPT,
    color='sample',
    palette=colors10,
    show=False,
    legend_loc='right margin',
    size=1,
    alpha=0.8,
    frameon=False,
    save='scgpt_sample_with_legend.svg'
)

plt.tight_layout()
plt.show()

/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


<Figure size 900x600 with 0 Axes>

### Ground truth annotation

In [55]:
# Plot UMAP by annnotation
sc.pl.umap(
    adata_scGPT,
    color='C_scANVI',
    palette=colors80,
    show=False,
    legend_loc=None,
    size=1,
    alpha=0.8,
    frameon=False,
    save='scgpt_C_scANVI_no_legend.svg'
)

plt.tight_layout()
plt.show()

/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


<Figure size 900x600 with 0 Axes>

In [56]:
# Plot UMAP by annnotation
sc.pl.umap(
    adata_scGPT,
    color='C_scANVI',
    palette=colors80,
    show=False,
    legend_loc='right margin',
    size=1,
    alpha=0.8,
    frameon=False,
    save='scgpt_C_scANVI_with_legend.svg'
)

plt.tight_layout()
plt.show()

/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


<Figure size 900x600 with 0 Axes>

### Grouped annotations

In [57]:
# Create a new column for highlighting
adata_scGPT.obs['highlight_spots'] = adata_scGPT.obs['C_scANVI'].astype(str)

#Brain regions
Neural_tube = [
    'Diencephalon_level2',
    'Hindbrain_level2',
    'Isthmus_level2',
    'Medulla oblongata_level2',
    'Midbrain_level1',
    'Pons_level2',
    'Rhombomere_level2',
    'Telencephalon_level2',
    'Spinal Cord_level0',
]

#Heart regions
heart = [
    'Left Atrium_level1',
    'Left Ventricle_level1',
    'Right Atrium_level1',
    'Right Ventricle_level1',
]

#Mark cells of interest
adata_scGPT.obs.loc[adata_scGPT.obs['highlight_spots'].isin(Neural_tube), 'highlight_spots'] = 'Neural tube'
adata_scGPT.obs.loc[adata_scGPT.obs['highlight_spots'].isin(heart), 'highlight_spots'] = 'Heart'
adata_scGPT.obs.loc[adata_scGPT.obs['highlight_spots'] == 'Liver_level0', 'highlight_spots'] = 'Liver'
adata_scGPT.obs.loc[~adata_scGPT.obs['highlight_spots'].isin(['Neural tube', 'Liver', 'Heart']), 'highlight_spots'] = 'Other'

# Create palette
highlight_palette = {
    'Other': '#D3D3D3',      
    'Liver': '#9966FF',  
    'Heart': '#CC0066', 
    'Neural tube': '#66B3FF'       
}

sc.pl.umap(
    adata_scGPT, 
    color='highlight_spots',
    palette=highlight_palette,
    show=False, 
    legend_loc=None,
    size=1,
    alpha=0.8,
    frameon=False,
    save='scGPT_highlight_spots_no_legend.svg'
)

plt.tight_layout()
plt.show()

/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


<Figure size 900x600 with 0 Axes>

In [58]:
sc.pl.umap(
    adata_scGPT, 
    color='highlight_spots',
    palette=highlight_palette,
    show=False, 
    legend_loc='right margin',
    size=1,
    alpha=0.8,
    frameon=False,
    save='scGPT_highlight_spots_with_legend.svg'
)

plt.tight_layout()
plt.show()

/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


<Figure size 900x600 with 0 Axes>

### Leiden clustering

In [59]:
#Leiden clustering
sc.tl.leiden(adata_scGPT, 
                 key_added=f'emb_leiden',
                 resolution=0.4)

# Plot UMAP with best resolution
sc.pl.umap(
    adata_scGPT, 
    color='emb_leiden',
    palette=colors80, 
    show=False, 
    legend_loc='right margin',
    size=1,
    alpha=0.8,
    frameon=False,
    save='scgpt_emb_leiden_with_legend.svg'
)

plt.tight_layout()
plt.show()

/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


<Figure size 900x600 with 0 Axes>

In [60]:
sc.pl.umap(
    adata_scGPT, 
    color='emb_leiden',
    palette=colors80, 
    show=False, 
    legend_loc=None,
    size=1,
    alpha=0.8,
    frameon=False,
    save='scgpt_emb_leiden_no_legend.svg'
)

plt.tight_layout()
plt.show()

/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/nfs/team361/mv10/.deep_speed_test/lib/python3.10/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


<Figure size 900x600 with 0 Axes>

In [61]:
# Leiden resolution with best ARI 
latent_leiden_resolution = 0.40

# Compute metrics over different resolutions
nmi_scores = []
ari_scores = []
n_cell_types_list = []

latent_leiden_resolution_list = [latent_leiden_resolution,
                                 latent_leiden_resolution-0.04,
                                 latent_leiden_resolution-0.03,
                                 latent_leiden_resolution-0.02,
                                 latent_leiden_resolution-0.01,
                                 latent_leiden_resolution+0.01,
                                 latent_leiden_resolution+0.02,
                                 latent_leiden_resolution+0.03,
                                 latent_leiden_resolution+0.04]

for i, latent_leiden_resolution in enumerate(latent_leiden_resolution_list[1:]):

    sc.tl.leiden(adata_scGPT,
                 key_added=f'emb_leiden',
                 resolution=latent_leiden_resolution)

    n_cell_types_list.append(adata_scGPT.obs[f'emb_leiden'].nunique())
    labels_true = adata_scGPT.obs['C_scANVI']
    labels_pred = adata_scGPT.obs['emb_leiden']
    
    nmi_score = normalized_mutual_info_score(labels_true, labels_pred, average_method="arithmetic")
    ari_score = adjusted_rand_score(labels_true, labels_pred)

    nmi_scores.append(nmi_score)
    ari_scores.append(ari_score)


print(f"NMI scores: {nmi_scores}.")
print(f"ARI scores: {ari_scores}.")
print(f"Number of clusters: {n_cell_types_list}.")

NMI scores: [0.3218774718802277, 0.3237745478235259, 0.31376092229295266, 0.3168252251559085, 0.3144779289411849, 0.3146048219360451, 0.31620186427859664, 0.3233860252269693].
ARI scores: [0.12375737251925624, 0.12371397000527021, 0.10901960669462464, 0.1120623810427388, 0.1090773661481145, 0.10561166249807813, 0.10993557058962002, 0.11594575963371824].
Number of clusters: [12, 13, 15, 14, 15, 16, 15, 16].
